# 질병별 조직 신호: 기존 cfRNA/cfDNA 문헌 vs NM

각 질병의 **관심 조직**(`EDA/disease_tissue_literature.md`, C1/C2 기준 재평가)에 대해
문헌(원 연구, 다른 cfRNA, cfDNA)과 NM이 그 조직 마커를 몇 개 잡았는지를 한 표/그림에 놓는다.
그룹 DEG는 쓰지 않는다: 보유한 DESeq2 결과가 원 연구의 분석(샘플셋, 방법, 공변량)을 재현한 것이 아니어서 기준선으로 부적절하다.
NM vs 그룹 비교는 Moore 매칭 비교(nb7)가 맡는다.

| 출처 | 정의 |
|---|---|
| 문헌 (Lit) | 해당 질병 cfRNA 논문이 그 조직 유래로 명시한 유전자 (md Part S의 [FT] 항목) |
| NM | 환자 1명 이상에서 BH q<0.05 상승, 같은 배치 HC에서는 0명 |
| NM 신규 | NM 유전자 중 문헌이 이름을 댄 마커가 아닌 것 |

- 조직 마커 패널(`notebookscripts/hpa_panels.py` `tct_panel_table`, HPA 25.1): 세포형 배정은 HPA tissue cell type(조직 안 세포형 enrichment),
  거기에 몸 전체 특이성(그 장기의 tissue specificity 또는 그 세포형의 single cell specificity; enriched / group enriched / enhanced)을 통과한 유전자만 남긴다.
  그 세포형의 single cell enriched 유전자는 tissue cell type에 빠져 있어도 포함한다. enriched 우선, 그다음 HPA 특이성 점수 순으로 패널당 최대 200개.
  HPA tissue cell type이 없는 패널(태반, 식도, 소장, 뇌, 혈소판)은 single cell enriched 규칙에 같은 상한.
  그다음 **혈장 특이성 필터**: 전체 HC pool에서 검출률 >= `PLASMA_MAX_HC_DETECT`(90%)인 유전자는 혈장에 원래 늘 있는 유전자로 보고 패널에서 뺀다.
- 문헌이 이름을 댄 마커 유전자는 패널 규칙·검출 필터와 무관하게 **항상** 표와 그림에 나온다.
- 원 연구가 조직을 지정하지 않은 문헌 유전자(`no tissue given`)도 같은 HPA 범주로 배정한다: enriched 세포형이 후보 패널에 있으면
  그중 nCPM이 가장 높은 패널 행으로 옮기고(`*`), 후보 밖 세포형(예: 호중구, T세포)이면 `unassigned` 행에 그 세포형을 괄호로 적는다.
  조직이 지정된 문헌 유전자를 HPA가 그 세포형으로 분류하지 않으면 `°`로 표시한다.
- 유전자 universe는 두 기준을 병기: `det10` = 비교 샘플 검출률 >= 10%, `read10` = 1개라도 검출 + 환자 hit의 read >= 10 (4절).
- 8절은 `read10`에서 **환자 1명에서만** 오른 조직 유전자를 사례 단위로 연다.
- 관심 조직 밖 패널(off-target)은 같은 계산을 거치는 내부 음성 대조로 남긴다.

주의: HC Z는 in-sample(학습에 쓰인 HC)이라 HC 쪽 편차가 과소평가될 수 있다.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import scipy.sparse as sp
import sys
import textwrap
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from scipy.stats import false_discovery_control, fisher_exact, norm

os.environ["NM_PIPELINE"] = "p0"
parent_dir = str(Path.cwd().parents[1])
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import FS_ANNOT_SM, apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.notebookscripts import hpa_panels
%matplotlib inline

OUT = config.PIPELINE_DIR / 'DiseaseTissue'
FIG = OUT / 'Figures'
FIG.mkdir(parents=True, exist_ok=True)
FDR_Q = 0.05
MIN_DETECT = 0.10
ENRICH_FOLD = 4
MIN_NCPM = 1
PLASMA_MAX_HC_DETECT = 0.9
PANEL_CAP = 200
OI = dict(orange='#E69F00', sky='#56B4E9', green='#009E73', yellow='#F0E442', blue='#0072B2',
          vermillion='#D55E00', purple='#CC79A7', black='#000000', grey='#999999', light='#D9D9D9')


def save_fig(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=300, bbox_inches='tight')

## 1. 관심 조직(prior)과 문헌 마커

문헌 근거의 출처는 `EDA/disease_tissue_literature.md` Part R.

**관심 조직 정의**
- `PRIOR`: 질병별 관심 조직. 표에 없는 조합은 off-target.
  - `keep` = C1 + C2 통과
  - `cond` = 조건부(임상 하위군 필요, 대부분 메타데이터 없음)
  - `blood` = 혈액 계통 참조 세트
- `TISSUE_OF`: 세포형 패널이 속한 장기.
- `RATIONALE`: 관심 조직으로 고른 이유 = (C1 세포 사멸/분비 기전, C2 임상 혈액 마커). C2의 다른 갈래인 cfDNA 근거는 `CFDNA`.
- `CONDITION`: `cond` 조직이 C1을 만족하려면 필요한 임상 상황.
  - 예: 위암의 간은 간전이 환자에서만 간세포 손상 기전이 있다.
  - 우리 메타데이터로 하위군을 실제로 가를 수 있는 경우만 `COND_IN_METADATA`(췌장암 조직형, 췌장염 유형).
  - 나머지는 전체 환자에 그대로 적용했으므로, 해당 행의 신호는 '조건을 만족하는 환자가 섞여 있다면 보일 수 있는 것'으로만 읽는다.

**문헌 근거**
- `SOURCE_STUDY`: 이 샘플을 만든 원 연구.
- `EVIDENCE`: 질병-조직별 문헌 근거를 세 갈래로 분리.
  - `source` = 원 연구가 보고 / `cfrna` = 다른 cfRNA 연구가 보고 / `cfdna` = cfDNA 조직 기원 연구가 보고
  - 값: `genes` = 유전자 이름까지, `tissue` = 조직·세포형 수준, `partial` = 일부 유전자만, 저자가 조직으로 해석하지 않음, 보충자료 미확인, 또는 HPA로 세포형을 배정함
- `CFDNA`: 같은 질병에서 cfDNA 조직 기원(메틸화) 연구가 그 조직 신호를 보고한 경우.
  - 출처: Lehmann-Werman 2016, Lubotzky 2022, Moss 2018, Sun 2015 / GETMap 2021, Lo 1999, van Vliet 2025
  - 다른 원발암의 간전이에서만 보인 경우는 넣지 않았다.

**문헌 마커 유전자 (`LIT_GENES`)**
- 원문에서 조직 유래로 명시된 유전자를 넣고, 원 연구의 마커 패널은 조직 귀속이 없어도 빠짐없이 넣는다.
- Moufarrej 2022의 18개
  - 출처: Extended Data Table 4 + 저자 코드 github.com/miramou/pe_cfrna `analysis/3.0_ML_Training.ipynb` (두 출처 일치)
  - 논문의 HPA 귀속대로 태반 2 / 신경근 4 / 귀속 없음 12(`no tissue given`, 면역 2 포함)
- Roskams-Hieter 2022
  - 출처: 보충자료 PDF(`41698_2022_270_MOESM1_ESM.pdf`) Supplementary Table 3(DESeq2 padj<0.01), Table 5(LVQ top 10)
  - HCC: 저자가 간 특이로 본 LVQ 유전자는 간세포 행, 나머지 DESeq2/LVQ 7개는 `no tissue given`
  - 간경변: DESeq2 21 + LVQ 10, 저자 조직 귀속 없음
  - MM: 골수(적혈구) 귀속 5개는 erythroid 행, LVQ 나머지 5개는 `no tissue given` (DESeq2 110개는 그림에 다 넣지 않음)
  - MGUS: DESeq2 1(TRADD) + LVQ 10, 귀속 없음
- `no tissue given` 유전자는 2절에서 HPA 공식 범주로 세포형을 배정한다(그림에서 `*`).
- `LIT_ENS`: 기호가 바뀐 유전자(FAM46A = TENT5A)와 Y_RNA 두 locus를 Ensembl ID로 맞추기 위한 표.
- `LIT_DOWN`: 원 연구에서 질병군에서 감소한 유전자. NM 집계는 상승만 센다.
- `LIT_TISSUE_ONLY`: 유전자 이름 없이 조직 수준으로만 보고된 경우.
  - md Part S의 [FT] 항목 전부: Moufarrej의 신경근·신장·신경·평활근, Tao 2023의 CAF·내피, Gardella의 기질 포함
- `LIT_NOTE`: 문헌이 '감소'로 보고했거나 방향이 엇갈린 경우. NM 비교는 상승만 세므로 '감소'로 보고된 조직에서 NM 0은 불일치가 아니다.

**급성기 단백 (`ACUTE_PHASE`)**
- 간에서 만들어지지만 간 손상이 아니라 전신 염증으로 오르는 단백.
  - 출처: Gabay & Kushner, NEJM 1999, 10.1056/NEJM199902113400607 (목록은 [recall] 기반이니 원문 Table 1과 대조할 것)
- 간세포 패널 상승이 이것들로만 채워지면 '간 조직 신호'가 아니라 염증 반응으로 읽는다.

In [ ]:
PANELS = {
    'hepatocyte': ['hepatocytes'],
    'cholangiocyte': ['cholangiocytes'],
    'hepatic stellate': ['hepatic stellate cells'],
    'pancreatic acinar': ['pancreatic acinar cells'],
    'pancreatic duct': ['pancreatic duct cells'],
    'pancreatic islet': ['pancreatic islet cells'],
    'colonocyte': ['colonocytes'],
    'enterocyte': ['enterocytes'],
    'gastric epithelium': ['gastric chief cells', 'foveolar cells', 'parietal cells', 'mucous neck cells'],
    'esophageal squamous': ['esophageal basal cells', 'esophageal suprabasal cells', 'esophageal apical cells'],
    'alveolar type 2': ['alveolar cells type 2'],
    'alveolar type 1': ['alveolar cells type 1'],
    'airway': ['respiratory basal cells', 'respiratory secretory cells', 'respiratory ciliated cells'],
    'macrophage': ['macrophages', 'kupffer cells', 'hofbauer cells', 'microglia'],
    'trophoblast': ['syncytiotrophoblasts', 'extravillous trophoblasts', 'cytotrophoblasts', 'migrating cytotrophoblasts'],
    'vascular endothelium': ['vascular endothelial cells'],
    'cardiomyocyte': ['cardiomyocytes'],
    'proximal tubule': ['proximal tubule cells'],
    'distal tubule / loop of Henle': ['distal convoluted tubule cells', 'loop of henle epithelial cells'],
    'skeletal muscle': ['myonuclei'],
    'plasma cell': ['plasma cells'],
    'erythroid': ['erythrocyte progenitors', 'erythrocytes'],
    'neutrophil': ['neutrophils', 'neutrophil progenitors'],
    'platelet / megakaryocyte': ['platelets', 'megakaryocytes', 'megakaryocyte progenitors'],
    'T cell': ['t-cells'],
    'fibroblast': ['fibroblasts'],
    'smooth muscle': ['smooth muscle cells', 'vascular smooth muscle cells'],
    'neuron': ['brain excitatory neurons', 'brain inhibitory neurons', 'other brain neurons'],
    'oligodendrocyte': ['oligodendrocytes'],
    'mesothelium': ['mesothelial cells'],
    'no tissue given': [],
}

PANEL_ADJACENT = {
    'pancreatic duct': ['cholangiocytes'],
    'cholangiocyte': ['pancreatic duct cells'],
    'macrophage': ['kupffer cells', 'hofbauer cells', 'microglia'],
    'alveolar type 2': ['transitional alveolar cells'],
    'alveolar type 1': ['transitional alveolar cells'],
    'fibroblast': ['fibro-adipogenic progenitors'],
    'colonocyte': ['enterocytes', 'goblet cells', 'enteric stem cells', 'enteric transient amplifying cells',
                   'tuft cells', 'paneth cells'],
    'enterocyte': ['colonocytes', 'goblet cells', 'enteric stem cells', 'enteric transient amplifying cells',
                   'tuft cells', 'paneth cells'],
}

CONDITION = {
    ('Pancreatic Cancer', 'cholangiocyte'): 'biliary obstruction', ('Pancreatic Cancer', 'pancreatic islet'): 'islet cell tumour',
    ('Pancreatitis', 'pancreatic islet'): 'chronic + diabetes', ('Pancreatitis', 'hepatocyte'): 'gallstone aetiology',
    ('Pancreatitis', 'proximal tubule'): 'severe acute (AKI)',
    ('Esophagus Cancer', 'esophageal squamous'): 'ESCC histology', ('Esophagus Cancer', 'gastric epithelium'): 'EAC histology',
    ('Esophagus Cancer', 'hepatocyte'): 'liver metastasis', ('Stomach Cancer', 'hepatocyte'): 'liver metastasis',
    ('Lung Cancer', 'alveolar type 1'): 'squamous / lung injury', ('Lung Cancer', 'airway'): 'squamous / lung injury',
    ('Lung Cancer', 'hepatocyte'): 'liver metastasis', ('Liver Cancer', 'proximal tubule'): 'AKI',
    ('Lung Cancer', 'vascular endothelium'): 'non-specific (also COPD)', ('Lung Cancer', 'neuron'): 'brain metastasis',
    ('Pre-eclampsia', 'neuron'): 'late gestation / eclampsia',
    ('Liver Cirrhosis', 'proximal tubule'): 'AKI', ('Liver Cirrhosis', 'enterocyte'): 'decompensated / alcohol',
    ('MM', 'proximal tubule'): 'renal impairment', ('MM', 'cardiomyocyte'): 'AL amyloidosis',
    ('Tuberculosis', 'hepatocyte'): 'on-treatment DILI', ('HIV + Tuberculosis', 'hepatocyte'): 'DILI / disseminated',
    ('CAD_HF+', 'cardiomyocyte'): 'raised hs-TnI', ('CAD_HF-', 'cardiomyocyte'): 'raised hs-TnI',
    ('Pre-eclampsia', 'hepatocyte'): 'HELLP', ('Pre-eclampsia', 'cardiomyocyte'): 'cardiac involvement',
    ('ME/CFS', 'skeletal muscle'): 'post-exertion sampling',
}
COND_IN_METADATA = {('Pancreatic Cancer', 'pancreatic islet'), ('Pancreatitis', 'pancreatic islet'),
                    ('Pancreatitis', 'proximal tubule')}

SOURCE_STUDY = {'Pancreatic Cancer': 'Moore 2025', 'Pancreatitis': 'Moore 2025', 'Colorectal Cancer': 'Chen 2022',
                'Esophagus Cancer': 'Chen 2022', 'Stomach Cancer': 'Chen 2022', 'Lung Cancer': 'Chen 2022',
                'Liver Cancer': 'Chen 2022; Roskams-Hieter 2022', 'Liver Cirrhosis': 'Roskams-Hieter 2022',
                'MM': 'Roskams-Hieter 2022', 'MGUS': 'Roskams-Hieter 2022', 'Tuberculosis': 'Chang 2024',
                'HIV': 'Chang 2024', 'HIV + Tuberculosis': 'Chang 2024', 'CAD_HF+': 'Ward 2022', 'CAD_HF-': 'Ward 2022',
                'Pre-eclampsia': 'Moufarrej 2022', 'ME/CFS': 'Gardella 2025'}
EVIDENCE = {
    ('Tuberculosis', 'neutrophil'): dict(cfdna='tissue'),
    ('Pancreatic Cancer', 'hepatocyte'): dict(source='genes', cfrna='partial', cfdna='tissue'),
    ('Pancreatic Cancer', 'pancreatic acinar'): dict(cfdna='tissue'),
    ('Pancreatitis', 'pancreatic acinar'): dict(cfdna='tissue'),
    ('Colorectal Cancer', 'hepatocyte'): dict(cfrna='partial', cfdna='tissue'),
    ('Colorectal Cancer', 'colonocyte'): dict(cfdna='tissue'),
    ('Lung Cancer', 'alveolar type 2'): dict(cfrna='genes', cfdna='tissue'),
    ('Lung Cancer', 'airway'): dict(cfrna='genes'),
    ('Lung Cancer', 'vascular endothelium'): dict(cfdna='tissue'),
    ('Liver Cancer', 'hepatocyte'): dict(source='genes', cfrna='tissue', cfdna='tissue'),
    ('Liver Cancer', 'no tissue given'): dict(source='genes'),
    ('Liver Cirrhosis', 'no tissue given'): dict(source='genes'),
    ('MM', 'no tissue given'): dict(source='genes'),
    ('MGUS', 'no tissue given'): dict(source='genes'),
    ('MM', 'erythroid'): dict(source='genes'),
    ('Tuberculosis', 'alveolar type 2'): dict(source='genes'),
    ('Tuberculosis', 'macrophage'): dict(source='genes'),
    ('HIV + Tuberculosis', 'alveolar type 2'): dict(source='genes'),
    ('HIV + Tuberculosis', 'macrophage'): dict(source='genes'),
    ('Pre-eclampsia', 'trophoblast'): dict(source='genes', cfrna='genes', cfdna='tissue'),
    ('Pre-eclampsia', 'skeletal muscle'): dict(source='genes'),
    ('Pre-eclampsia', 'no tissue given'): dict(source='genes'),
    ('Pre-eclampsia', 'vascular endothelium'): dict(source='tissue', cfdna='tissue'),
    ('Pre-eclampsia', 'hepatocyte'): dict(source='tissue'),
    ('Pre-eclampsia', 'neuron'): dict(source='tissue'),
}
EVIDENCE_REF = {
    'source': 'the study that produced these samples (SOURCE_STUDY)',
    'cfrna': 'other cfRNA: Yu 2020 (PDAC EV, FGA/ITIH2); PLoS One 2024 (CRC, FGA/FGB); Larson 2021, Nesselbush 2025 (lung); '
             'Sayeed 2020 (HCC); Tsang 2017, Rasmussen 2022 (PE trophoblast)',
    'cfdna': 'cfDNA methylation: Lubotzky 2022, Lehmann-Werman 2016, Moss 2018, Peretz 2023, Sun 2015 / GETMap 2021, '
             'Lo 1999, van Vliet 2025',
}

RATIONALE = {
    ('Pancreatic Cancer', 'hepatocyte'): ('biliary obstruction / liver mets', 'ALT/AST, bilirubin'),
    ('Pancreatic Cancer', 'cholangiocyte'): ('biliary obstruction', 'ALP, GGT'),
    ('Pancreatic Cancer', 'pancreatic acinar'): ('duct obstruction, acinar atrophy', 'lipase, amylase'),
    ('Pancreatic Cancer', 'pancreatic duct'): ('tumour cell turnover', '-'),
    ('Pancreatic Cancer', 'pancreatic islet'): ('islet cell tumour itself', 'chromogranin A'),
    ('Pancreatitis', 'pancreatic acinar'): ('acinar necrosis / fibrosis', 'lipase, amylase'),
    ('Pancreatitis', 'pancreatic islet'): ('islet loss in chronic disease', 'glucose, HbA1c'),
    ('Pancreatitis', 'hepatocyte'): ('gallstone biliary injury', 'ALT, bilirubin'),
    ('Pancreatitis', 'proximal tubule'): ('AKI in severe acute disease', 'creatinine'),
    ('Colorectal Cancer', 'colonocyte'): ('tumour cell turnover', '-'),
    ('Colorectal Cancer', 'hepatocyte'): ('liver (micro)metastasis', 'ALT/AST, ALP'),
    ('Esophagus Cancer', 'esophageal squamous'): ('tumour cell turnover (ESCC)', '-'),
    ('Esophagus Cancer', 'gastric epithelium'): ('tumour cell turnover (EAC)', '-'),
    ('Esophagus Cancer', 'hepatocyte'): ('liver metastasis', 'ALT/AST, ALP'),
    ('Stomach Cancer', 'gastric epithelium'): ('tumour / mucosal turnover', 'pepsinogen I/II'),
    ('Stomach Cancer', 'hepatocyte'): ('liver metastasis', 'ALT/AST, ALP'),
    ('Lung Cancer', 'alveolar type 2'): ('tumour (LUAD) / lung injury', 'KL-6, SP-D'),
    ('Lung Cancer', 'alveolar type 1'): ('lung injury', 'KL-6'),
    ('Lung Cancer', 'airway'): ('tumour (LUSC) / airway injury', '-'),
    ('Lung Cancer', 'hepatocyte'): ('liver metastasis', 'ALT/AST'),
    ('Lung Cancer', 'vascular endothelium'): ('tumour lung vascular turnover', '-'),
    ('Lung Cancer', 'neuron'): ('brain metastasis', 'NfL, GFAP'),
    ('Liver Cancer', 'hepatocyte'): ('tumour + cirrhotic liver injury', 'ALT/AST, ALBI'),
    ('Liver Cancer', 'proximal tubule'): ('AKI / hepatorenal', 'creatinine'),
    ('Liver Cirrhosis', 'hepatocyte'): ('hepatocyte death', 'ALT/AST'),
    ('Liver Cirrhosis', 'hepatic stellate'): ('fibrogenesis', 'ELF (HA, PIIINP, TIMP1)'),
    ('Liver Cirrhosis', 'proximal tubule'): ('AKI / hepatorenal', 'creatinine'),
    ('Liver Cirrhosis', 'enterocyte'): ('gut barrier injury', 'I-FABP'),
    ('MM', 'plasma cell'): ('clonal plasma cells (blood lineage)', 'M-protein, FLC'),
    ('MM', 'erythroid'): ('marrow erythropoiesis (blood lineage)', 'haemoglobin'),
    ('MM', 'proximal tubule'): ('cast nephropathy', 'creatinine'),
    ('MM', 'cardiomyocyte'): ('AL amyloid infiltration', 'troponin, NT-proBNP'),
    ('MGUS', 'plasma cell'): ('clonal plasma cells (blood lineage)', 'M-protein, FLC'),
    ('MGUS', 'erythroid'): ('marrow erythropoiesis (blood lineage)', 'haemoglobin'),
    ('Tuberculosis', 'alveolar type 2'): ('parenchymal destruction', 'SP-A, SP-D'),
    ('Tuberculosis', 'macrophage'): ('granuloma turnover', '-'),
    ('Tuberculosis', 'hepatocyte'): ('drug-induced liver injury', 'ALT/AST'),
    ('HIV', 'enterocyte'): ('GALT / enterocyte damage', 'I-FABP, sCD14'),
    ('HIV + Tuberculosis', 'alveolar type 2'): ('parenchymal destruction', 'SP-A, SP-D'),
    ('HIV + Tuberculosis', 'macrophage'): ('granuloma turnover', '-'),
    ('HIV + Tuberculosis', 'enterocyte'): ('GALT / enterocyte damage', 'I-FABP, sCD14'),
    ('HIV + Tuberculosis', 'hepatocyte'): ('drug injury / dissemination', 'ALT/AST'),
    ('CAD_HF+', 'vascular endothelium'): ('endothelial apoptosis', 'endothelial microparticles'),
    ('CAD_HF+', 'cardiomyocyte'): ('chronic myocyte injury', 'hs-TnT/I'),
    ('CAD_HF-', 'vascular endothelium'): ('endothelial apoptosis', 'endothelial microparticles'),
    ('CAD_HF-', 'cardiomyocyte'): ('chronic myocyte injury', 'hs-TnT/I'),
    ('Pre-eclampsia', 'trophoblast'): ('syncytiotrophoblast stress', 'sFlt-1/PlGF'),
    ('Pre-eclampsia', 'vascular endothelium'): ('endothelial damage', 'sFlt-1, circulating ECs'),
    ('Pre-eclampsia', 'hepatocyte'): ('HELLP liver injury', 'ALT/AST, LDH'),
    ('Pre-eclampsia', 'cardiomyocyte'): ('cardiac strain', 'cTnI'),
    ('Pre-eclampsia', 'neuron'): ('cerebral injury', 'NfL'),
    ('ME/CFS', 'skeletal muscle'): ('post-exertional muscle injury', 'CK'),
    ('Pre-eclampsia', 'skeletal muscle'): ('literature only: neuromuscular (HPA)', '-'),
    ('Pancreatitis', 'alveolar type 2'): ('ARDS in severe acute disease', 'PaO2/FiO2'),
    ('Pancreatitis', 'alveolar type 1'): ('ARDS in severe acute disease', 'PaO2/FiO2'),
    ('Liver Cancer', 'hepatic stellate'): ('background cirrhosis fibrogenesis', 'ELF (HA, PIIINP, TIMP1)'),
    ('Liver Cancer', 'macrophage'): ('Kupffer / scar macrophage activation', 'sCD163'),
    ('Liver Cancer', 'cholangiocyte'): ('ductular reaction / cholestasis', 'ALP, GGT'),
    ('Liver Cirrhosis', 'macrophage'): ('Kupffer / scar macrophage activation', 'sCD163'),
    ('Liver Cirrhosis', 'cholangiocyte'): ('ductular reaction', 'ALP, GGT'),
    ('MM', 'distal tubule / loop of Henle'): ('cast nephropathy', 'creatinine'),
    ('Tuberculosis', 'alveolar type 1'): ('parenchymal necrosis', 'KL-6'),
    ('Tuberculosis', 'airway'): ('endobronchial erosion', 'CC16'),
    ('Tuberculosis', 'mesothelium'): ('pleural TB', '-'),
    ('Tuberculosis', 'neutrophil'): ('NETs / neutrophil death (blood lineage)', 'MPO-DNA'),
    ('HIV', 'T cell'): ('CD4 T-cell depletion (blood lineage)', 'CD4 count'),
    ('HIV', 'vascular endothelium'): ('endothelial activation', 'sVCAM-1, sICAM-1'),
    ('HIV', 'hepatocyte'): ('ART hepatotoxicity / coinfection', 'ALT/AST'),
    ('HIV', 'alveolar type 2'): ('intercurrent pneumonia', 'SP-D'),
    ('HIV + Tuberculosis', 'alveolar type 1'): ('parenchymal necrosis', 'KL-6'),
    ('HIV + Tuberculosis', 'mesothelium'): ('pleural TB', '-'),
    ('HIV + Tuberculosis', 'vascular endothelium'): ('endothelial activation', 'sVCAM-1'),
    ('CAD_HF+', 'platelet / megakaryocyte'): ('platelet activation (blood lineage)', 'platelet count'),
    ('CAD_HF-', 'platelet / megakaryocyte'): ('platelet activation (blood lineage)', 'platelet count'),
    ('Pre-eclampsia', 'oligodendrocyte'): ('neuroaxonal injury', 'NfL'),
    ('Pre-eclampsia', 'platelet / megakaryocyte'): ('platelet consumption (blood lineage)', 'platelet count'),
}

TISSUE_OF = {'hepatocyte': 'Liver', 'cholangiocyte': 'Liver', 'hepatic stellate': 'Liver',
             'pancreatic acinar': 'Pancreas', 'pancreatic duct': 'Pancreas', 'pancreatic islet': 'Pancreas',
             'colonocyte': 'Colon', 'enterocyte': 'Small intestine', 'gastric epithelium': 'Stomach',
             'esophageal squamous': 'Esophagus', 'alveolar type 2': 'Lung', 'alveolar type 1': 'Lung',
             'airway': 'Lung', 'macrophage': 'Tissue macrophage', 'trophoblast': 'Placenta',
             'vascular endothelium': 'Vasculature', 'cardiomyocyte': 'Heart', 'proximal tubule': 'Kidney',
             'skeletal muscle': 'Skeletal muscle', 'plasma cell': 'Blood / marrow', 'erythroid': 'Blood / marrow',
             'fibroblast': 'Stroma', 'smooth muscle': 'Smooth muscle', 'neuron': 'Brain', 'no tissue given': '-',
             'distal tubule / loop of Henle': 'Kidney', 'neutrophil': 'Blood / marrow',
             'platelet / megakaryocyte': 'Blood / marrow', 'T cell': 'Blood / marrow', 'oligodendrocyte': 'Brain',
             'mesothelium': 'Pleura / peritoneum'}

PRIOR = {
    'Pancreatic Cancer': {'pancreatic duct': 'keep', 'pancreatic acinar': 'keep', 'hepatocyte': 'keep',
                          'cholangiocyte': 'cond', 'pancreatic islet': 'cond'},
    'Pancreatitis': {'pancreatic acinar': 'keep', 'pancreatic islet': 'cond', 'hepatocyte': 'cond',
                     'proximal tubule': 'cond', 'alveolar type 2': 'cond', 'alveolar type 1': 'cond'},
    'Colorectal Cancer': {'colonocyte': 'keep', 'hepatocyte': 'keep'},
    'Esophagus Cancer': {'esophageal squamous': 'cond', 'gastric epithelium': 'cond', 'hepatocyte': 'cond'},
    'Stomach Cancer': {'gastric epithelium': 'keep', 'hepatocyte': 'cond'},
    'Lung Cancer': {'alveolar type 2': 'keep', 'alveolar type 1': 'cond', 'airway': 'cond', 'hepatocyte': 'cond',
                    'vascular endothelium': 'cond', 'neuron': 'cond'},
    'Liver Cancer': {'hepatocyte': 'keep', 'proximal tubule': 'cond', 'hepatic stellate': 'cond',
                     'macrophage': 'cond', 'cholangiocyte': 'cond'},
    'Liver Cirrhosis': {'hepatocyte': 'keep', 'hepatic stellate': 'keep', 'proximal tubule': 'cond',
                        'enterocyte': 'cond', 'macrophage': 'cond', 'cholangiocyte': 'cond'},
    'MM': {'plasma cell': 'blood', 'erythroid': 'blood', 'proximal tubule': 'cond', 'cardiomyocyte': 'cond',
           'distal tubule / loop of Henle': 'cond'},
    'MGUS': {'plasma cell': 'blood', 'erythroid': 'blood'},
    'Tuberculosis': {'alveolar type 2': 'keep', 'macrophage': 'keep', 'hepatocyte': 'cond', 'alveolar type 1': 'keep',
                     'airway': 'cond', 'mesothelium': 'cond', 'neutrophil': 'blood'},
    'HIV': {'enterocyte': 'keep', 'T cell': 'blood', 'vascular endothelium': 'cond', 'hepatocyte': 'cond',
            'alveolar type 2': 'cond'},
    'HIV + Tuberculosis': {'alveolar type 2': 'keep', 'macrophage': 'keep', 'enterocyte': 'keep', 'hepatocyte': 'cond',
                           'alveolar type 1': 'cond', 'mesothelium': 'cond', 'vascular endothelium': 'cond'},
    'CAD_HF+': {'vascular endothelium': 'keep', 'cardiomyocyte': 'cond', 'platelet / megakaryocyte': 'blood'},
    'CAD_HF-': {'vascular endothelium': 'keep', 'cardiomyocyte': 'cond', 'platelet / megakaryocyte': 'blood'},
    'Pre-eclampsia': {'trophoblast': 'keep', 'vascular endothelium': 'keep', 'hepatocyte': 'cond', 'neuron': 'cond',
                      'cardiomyocyte': 'cond', 'oligodendrocyte': 'cond', 'platelet / megakaryocyte': 'blood'},
    'ME/CFS': {'skeletal muscle': 'cond'},
}

LIT_GENES = {
    ('Pancreatic Cancer', 'hepatocyte'): (['SERPINA1', 'CRP', 'ACOT12', 'AFMID'], 'Moore 2025'),
    ('Liver Cancer', 'hepatocyte'): (['APOE', 'C3', 'CP', 'DHCR24', 'FGA', 'FGB', 'FGG', 'HRG', 'TF'],
                                     'Roskams-Hieter 2022; Chen 2022'),
    ('Liver Cancer', 'no tissue given'): (['IFITM3', 'ATP1B1', 'APOH', 'HULC', 'UGT2B15', 'VTN', 'CAPRIN2'],
                                          'Roskams-Hieter 2022'),
    ('Liver Cirrhosis', 'no tissue given'): (['AGL', 'FCGR2A', 'RAP1GAP', 'OSBP2', 'GNS', 'LRP1', 'SCML1', 'IFITM3',
                                              'L3MBTL4', 'CD96', 'EIF1', 'METTL9', 'TAPBP', 'DAP', 'TRAC', 'LEF1', 'CD3D',
                                              'NSRP1', 'SYF2', 'PIK3R1', 'RPL36A', 'ABCB7', 'HIST1H2BF', 'PSIP1',
                                              'TMEM150C', 'ZC3H6', 'C9orf16', 'CPQ', 'DYNC1I2', 'ECM1', 'HIST1H2AE'],
                                             'Roskams-Hieter 2022'),
    ('MM', 'no tissue given'): (['CENPE', 'NUSAP1', 'NEK2', 'ELL2', 'AIDA'], 'Roskams-Hieter 2022'),
    ('MGUS', 'no tissue given'): (['TRADD', 'FPR3', 'SMC4', 'TXNDC16', 'VCAM1', 'ASPM', 'DNAH1', 'AL121603.2',
                                   'FAM109B', 'FMN1'], 'Roskams-Hieter 2022'),
    ('Lung Cancer', 'alveolar type 2'): (['SFTA3', 'SFTPA2', 'SLC34A2'], 'Larson 2021'),
    ('Lung Cancer', 'airway'): (['BPIFA1'], 'Larson 2021'),
    ('Tuberculosis', 'alveolar type 2'): (['SFTPC', 'SFTPB', 'SLC34A2', 'SFTPA1', 'SFTPA2'], 'Chang 2024'),
    ('Tuberculosis', 'macrophage'): (['MARCO', 'SOCS3', 'FCGR1A', 'MPO', 'C1QB'], 'Chang 2024'),
    ('HIV + Tuberculosis', 'alveolar type 2'): (['SFTPC', 'SFTPB', 'SLC34A2', 'SFTPA1', 'SFTPA2'], 'Chang 2024'),
    ('HIV + Tuberculosis', 'macrophage'): (['MARCO', 'SOCS3', 'FCGR1A', 'MPO', 'C1QB'], 'Chang 2024'),
    ('MM', 'erythroid'): (['CA1', 'EPB42', 'HBG1', 'HBG2', 'CPOX'], 'Roskams-Hieter 2022'),
    ('Pre-eclampsia', 'trophoblast'): (['TENT5A', 'MYLIP'], 'Moufarrej 2022'),
    ('Pre-eclampsia', 'skeletal muscle'): (['CAMK2G', 'NDUFV3', 'PI4KA', 'PRTFDC1'], 'Moufarrej 2022'),
    ('Pre-eclampsia', 'no tissue given'): (['RNF149', 'TRIM21', 'DERA', 'KIAA1109', 'LRRC58', 'NMRK1', 'PYGO2', 'TFIP11',
                                            'USB1', 'Y_RNA-1', 'Y_RNA-2', 'YWHAQP5'], 'Moufarrej 2022'),
}
LIT_ENS = {'TENT5A': 'ENSG00000112773', 'MYLIP': 'ENSG00000007944', 'CAMK2G': 'ENSG00000148660',
           'NDUFV3': 'ENSG00000160194', 'PI4KA': 'ENSG00000241973', 'PRTFDC1': 'ENSG00000099256',
           'RNF149': 'ENSG00000163162', 'TRIM21': 'ENSG00000132109', 'DERA': 'ENSG00000023697',
           'KIAA1109': 'ENSG00000138688', 'LRRC58': 'ENSG00000163428', 'NMRK1': 'ENSG00000106733',
           'PYGO2': 'ENSG00000163348', 'TFIP11': 'ENSG00000100109', 'USB1': 'ENSG00000103005',
           'Y_RNA-1': 'ENSG00000201412', 'Y_RNA-2': 'ENSG00000238912', 'YWHAQP5': 'ENSG00000236564'}
LIT_DOWN = {'TRAC', 'LEF1', 'CD3D', 'NSRP1', 'SYF2', 'PIK3R1', 'RPL36A', 'CAMK2G', 'TENT5A', 'KIAA1109', 'LRRC58', 'MYLIP', 'NDUFV3', 'PI4KA', 'PYGO2', 'RNF149', 'TFIP11',
            'TRIM21', 'USB1'}
LIT_TISSUE_ONLY = {('Pre-eclampsia', 'trophoblast'), ('Pre-eclampsia', 'vascular endothelium'),
                   ('Pre-eclampsia', 'hepatocyte'), ('Pre-eclampsia', 'skeletal muscle'),
                   ('Pre-eclampsia', 'proximal tubule'), ('Pre-eclampsia', 'neuron'), ('Pre-eclampsia', 'smooth muscle'),
                   ('Colorectal Cancer', 'hepatocyte'),
                   ('Colorectal Cancer', 'fibroblast'), ('Colorectal Cancer', 'vascular endothelium'),
                   ('Stomach Cancer', 'fibroblast'), ('Stomach Cancer', 'vascular endothelium'),
                   ('ME/CFS', 'fibroblast')}
CFDNA = {('Pancreatic Cancer', 'hepatocyte'), ('Pancreatic Cancer', 'pancreatic acinar'),
         ('Pancreatitis', 'pancreatic acinar'), ('Colorectal Cancer', 'colonocyte'), ('Colorectal Cancer', 'hepatocyte'),
         ('Lung Cancer', 'alveolar type 2'), ('Liver Cancer', 'hepatocyte'),
         ('Pre-eclampsia', 'trophoblast'), ('Pre-eclampsia', 'vascular endothelium'),
         ('Lung Cancer', 'vascular endothelium')}
LIT_NOTE = {('Pre-eclampsia', 'hepatocyte'): 'lit: down in PE', ('Pre-eclampsia', 'neuron'): 'lit: Moufarrej 2022',
            ('Pre-eclampsia', 'trophoblast'): 'lit: down in PE', ('Pre-eclampsia', 'proximal tubule'): 'lit: down in PE',
            ('Pre-eclampsia', 'smooth muscle'): 'lit: direction conflicts', ('Pre-eclampsia', 'skeletal muscle'): 'lit: neuromuscular',
            ('ME/CFS', 'fibroblast'): 'lit: higher in controls', ('Colorectal Cancer', 'fibroblast'): 'lit: Tao 2023 (CAF)',
            ('Stomach Cancer', 'fibroblast'): 'lit: Tao 2023 (CAF)', ('Colorectal Cancer', 'vascular endothelium'): 'lit: Tao 2023',
            ('Stomach Cancer', 'vascular endothelium'): 'lit: Tao 2023', ('Colorectal Cancer', 'hepatocyte'): 'lit: FGA/FGB only'}

ACUTE_PHASE = {'CRP', 'SAA1', 'SAA2', 'SAA4', 'HP', 'HPR', 'HPX', 'ORM1', 'ORM2', 'FGA', 'FGB', 'FGG',
               'SERPINA1', 'SERPINA3', 'SPINK1', 'ITIH1', 'ITIH2', 'ITIH3', 'ITIH4', 'CP', 'C3', 'C4A', 'C4B', 'C9',
               'MBL2', 'CFB', 'SERPING1', 'C4BPA', 'C4BPB', 'PLG', 'SERPINE1', 'VTN', 'PROS1', 'PLA2G2A', 'LBP',
               'IL1RN', 'FN1', 'AGT', 'APCS', 'HAMP', 'LCN2', 'FTL', 'FTH1'}

## 2. HPA 조직 마커 패널

In [ ]:
hpa = pd.read_csv(config.HPA_SINGLE_CELL_ZIP, sep='\t')
mat = hpa.pivot_table(index='Gene', columns='Cell type', values='nCPM', aggfunc='max')
hpa_sym = hpa.drop_duplicates('Gene').set_index('Gene')['Gene name']

ZD = config.ZSCORES_MIXED_DIR
genes = np.array(pickle.load(open(ZD / 'gene_names.pkl', 'rb')))
ens0 = pd.Index([g.split('.')[0] for g in genes])

rows = []
for p, types in PANELS.items():
    if not types:
        continue
    inside = mat[types].max(axis=1)
    outside = mat.drop(columns=types + PANEL_ADJACENT.get(p, [])).max(axis=1)
    ok = (inside >= MIN_NCPM) & (inside >= ENRICH_FOLD * outside)
    rows += [dict(panel=p, ens=g, symbol=hpa_sym[g], ncpm_in=inside[g], ncpm_out=outside[g]) for g in ok[ok].index]
panel_old = pd.DataFrame(rows)

atlas = pd.read_csv(config.HPA_ATLAS_ZIP, sep='\t', usecols=['Gene', 'Ensembl', 'RNA single cell type specificity',
                                                             'RNA single cell type specific nCPM'])
atlas = atlas[atlas['RNA single cell type specificity'].isin(['Cell type enriched', 'Group enriched'])].set_index('Ensembl')
spec = atlas['RNA single cell type specific nCPM'].map(
    lambda s: {t.split(': ')[0].lower(): float(t.split(': ')[1]) for t in s.split(';')})
panel_of_type = {t: p for p, ts in PANELS.items() for t in ts}
rows = [dict(panel=p, ens=e, symbol=atlas.at[e, 'Gene'], hpa_category=atlas.at[e, 'RNA single cell type specificity'],
             hpa_types=';'.join(d), ncpm_in=max(v for t, v in d.items() if panel_of_type.get(t) == p))
        for e, d in spec.items() for p in {panel_of_type[t] for t in d if t in panel_of_type}]
panel_tbl = hpa_panels.tct_panel_table(PANELS, cap=PANEL_CAP)
panel_tbl = panel_tbl[panel_tbl['ens'].isin(ens0)].assign(col=lambda d: ens0.get_indexer(d['ens']))
panel_tbl['gene'] = genes[panel_tbl['col']]
panel_tbl.to_csv(OUT / 'panel_genes.csv', index=False)

po = panel_old[panel_old['ens'].isin(ens0)]
both = panel_tbl.merge(po[['panel', 'ens']], on=['panel', 'ens'])
print(pd.DataFrame({'HPA panel': panel_tbl.groupby('panel').size(), 'own 4x rule': po.groupby('panel').size(),
                    'shared': both.groupby('panel').size()}).reindex(PANELS).dropna(how='all').fillna(0).astype(int).to_string())

ens_of_sym = pd.Series(atlas.index, index=atlas['Gene'])
ens_of_sym = ens_of_sym[~ens_of_sym.index.duplicated()]
hpa_types = lambda x: spec.get(LIT_ENS.get(x) or ens_of_sym.get(x), {})
LIT_MOVED = set()
for (ph, p), (gs, src) in list(LIT_GENES.items()):
    if p != 'no tissue given':
        continue
    keep = []
    for x in gs:
        d = {t: v for t, v in hpa_types(x).items() if t in panel_of_type}
        if not d:
            keep.append(x)
            continue
        q = panel_of_type[max(d, key=d.get)]
        old_gs, old_src = LIT_GENES.get((ph, q), ([], ''))
        LIT_GENES[(ph, q)] = (old_gs + [x], '; '.join(dict.fromkeys(filter(None, old_src.split('; ') + [src]))))
        EVIDENCE.setdefault((ph, q), {}).setdefault('source', 'partial')
        LIT_MOVED.add((ph, q, x))
    LIT_GENES[(ph, p)] = (keep, src)
print('literature genes moved from no tissue given by HPA category:', sorted(LIT_MOVED))

## 3. NM 상승 편차와 비교 샘플

- NM 환자: OOD 통과 + 같은 배치 HC가 `MIN_HC_BATCH_SIZE` 이상인 배치. HC: 그 배치들의 HC.
- 샘플별 BH는 engine 전체 유전자에 대해 한 번 계산(nb7 `bh_sig_matrix`와 동일), 상승(Z>0)만 사용.

In [ ]:
dm = pd.read_csv(ZD / 'sample_meta.csv')
hm = pd.read_csv(ZD / 'hc_meta.csv')
names = np.concatenate([dm['sample'], hm['sample']])
row_of = pd.Series(np.arange(len(names)), index=names)

UP_CACHE = OUT / 'nm_up_bh.npy'
if UP_CACHE.exists():
    UP = np.load(UP_CACHE)
else:
    Z = np.vstack([np.load(ZD / 'Z_disease_shash.npy'), np.load(ZD / 'Z_hc_shash.npy')])
    UP = np.vstack([bh_fdr_reject(2 * norm.sf(np.abs(z)), q=FDR_Q) & (z > 0) for z in Z])
    np.save(UP_CACHE, UP)

hc_n = hm.groupby('batch').size()
good = set(hc_n[hc_n >= config.MIN_HC_BATCH_SIZE].index)
cohort = {}
for ph in PRIOR:
    case = dm[(dm['phenotype'] == ph) & dm['ood_keep'] & dm['batch'].isin(good)]
    hc = hm[hm['batch'].isin(set(case['batch']))]
    cohort[ph] = dict(case=case['sample'].values, hc=hc['sample'].values, batches=sorted(set(case['batch'])))
print(pd.DataFrame({p: dict(n_case=len(c['case']), n_hc=len(c['hc']), n_batch=len(c['batches']))
                    for p, c in cohort.items()}).T.to_string())

In [ ]:
lit_syms = sorted({s for g, _ in LIT_GENES.values() for s in g})
var_sym = ad.read_h5ad(config.H5AD_PATH, backed='r').var['GeneName'].astype(str)
sym_of_gene = var_sym.reindex(genes)
gpos_ens = pd.Series(genes, index=ens0)
lit_id_of = {}
for x in lit_syms:
    if LIT_ENS.get(x) in gpos_ens.index:
        lit_id_of[x] = gpos_ens[LIT_ENS[x]]
    elif (sym_of_gene == x).any():
        lit_id_of[x] = sym_of_gene.index[sym_of_gene == x][0]
lit_gene_ids = sorted(set(lit_id_of.values()))
lit_unmodeled = sorted(set(lit_syms) - set(lit_id_of))
print('literature markers not modelled by the engine:', lit_unmodeled)
check_genes = sorted(set(panel_tbl['gene']) | set(lit_gene_ids))
need = sorted(set(np.concatenate([c['case'] for c in cohort.values()])) | set(hm['sample']))

CNT_CACHE = OUT / 'panel_counts.pkl'
cnt = None
if CNT_CACHE.exists():
    cnt, libsize = pickle.load(open(CNT_CACHE, 'rb'))
if cnt is None or not set(check_genes) <= set(cnt.columns):
    A = ad.read_h5ad(config.H5AD_PATH, backed='r')
    si = A.obs_names.get_indexer(need)
    order = np.argsort(si)
    X = A.X[si[order]][:, A.var_names.get_indexer(check_genes)].toarray()
    cnt = pd.DataFrame(X, index=np.array(need)[order], columns=check_genes).loc[need]
    libsize = A.obs['total_counts'].reindex(need).astype(float)
    pickle.dump((cnt, libsize), open(CNT_CACHE, 'wb'))
cpm = cnt.div(libsize, axis=0) * 1e6
det = pd.DataFrame({ph: (cnt.loc[np.concatenate([c['case'], c['hc']])] > 0).mean() for ph, c in cohort.items()})

panel_tbl['hc_pool_detect'] = (cnt.loc[hm['sample'], panel_tbl['gene']] > 0).mean().values
panel_all = panel_tbl.copy()
panel_tbl = panel_tbl[panel_tbl['hc_pool_detect'] < PLASMA_MAX_HC_DETECT].copy()
print('plasma-ubiquitous genes removed per panel (HC pool detection >= '
      f'{PLASMA_MAX_HC_DETECT:.0%}):')
print(pd.DataFrame({'before': panel_all.groupby('panel').size(), 'after': panel_tbl.groupby('panel').size()})
      .reindex(PANELS).fillna(0).astype(int).T.to_string())

Z_all = np.vstack([np.load(ZD / 'Z_disease_shash.npy', mmap_mode='r'), np.load(ZD / 'Z_hc_shash.npy', mmap_mode='r')])
n_up = pd.Series(UP.sum(1), index=names)

## 4. 질병 x 조직 비교표 -- 두 기준 병기

| 기준 | 유전자 포함 | 환자 hit | HC hit |
|---|---|---|---|
| `det10` | 비교 샘플 검출률 >= 10% | BH 상승 | BH 상승 |
| `read10` | 1개 샘플이라도 검출 | BH 상승 **이고** 그 환자의 read >= `MIN_HIT_READS` | BH 상승 (read 조건 없음, 보수적) |

`read10`은 희소 유전자를 살리되 read 1~3개짜리 hit는 버린다. 1명 hit(`n_pat == 1`)는 대부분 이 기준에서 새로 생긴다.

In [ ]:
MIN_HIT_READS = 10


def summarize(rule):
    rows, gene_rows = [], []
    for ph, c in cohort.items():
        P, H = UP[row_of[c['case']].values], UP[row_of[c['hc']].values]
        for p in PANELS:
            lit, src = LIT_GENES.get((ph, p), ([], ''))
            lit_nm = [x for x in lit if x in lit_id_of]
            lit_ids = [lit_id_of[x] for x in lit_nm]
            g = pd.concat([pd.DataFrame(dict(gene=lit_ids, col=pd.Index(genes).get_indexer(lit_ids), symbol=lit_nm)),
                           panel_tbl.loc[panel_tbl['panel'] == p, ['gene', 'col', 'symbol']]]).drop_duplicates('gene')
            is_lit0 = g['symbol'].isin(lit).values
            g = g[(det.loc[g['gene'], ph].values >= (MIN_DETECT if rule == 'det10' else 1e-9)) | is_lit0]
            Pg = P[:, g['col']]
            if rule == 'read10':
                Pg = Pg & (cnt.loc[c['case'], g['gene']].values >= MIN_HIT_READS)
            Hg = H[:, g['col']]
            in_panel = g['gene'].isin(panel_tbl.loc[panel_tbl['panel'] == p, 'gene']).values
            pn, hn = Pg.sum(0), Hg.sum(0)
            nm = (pn > 0) & (hn == 0)
            is_lit = g['symbol'].isin(lit).values
            new = nm & in_panel & ~is_lit
            a, b = int(Pg[:, in_panel].any(1).sum()), int(Hg[:, in_panel].any(1).sum())
            rows.append(dict(rule=rule, phenotype=ph, panel=p, prior=PRIOR[ph].get(p, 'off'),
                             lit_level='genes' if lit else ('tissue only' if (ph, p) in LIT_TISSUE_ONLY else 'none'),
                             lit_source=src, n_case=len(P), n_hc=len(H), n_panel=int(in_panel.sum()),
                             lit_n=len(lit), lit_detected=int((is_lit & (det.loc[g['gene'], ph].values > 0)).sum()),
                             lit_by_nm=int((is_lit & nm).sum()), nm_up=int((nm & in_panel).sum()),
                             nm_new=int(new.sum()), nm_new_single=int((new & (pn == 1)).sum()),
                             nm_new_non_ap=int((new & ~g['symbol'].isin(ACUTE_PHASE).values).sum()),
                             pat_carrier=a / len(P), hc_carrier=b / len(H),
                             fisher_p=fisher_exact([[a, len(P) - a], [b, len(H) - b]], alternative='greater')[1]))
            for j, (s, gn) in enumerate(zip(g['symbol'], g['gene'])):
                gene_rows.append(dict(rule=rule, phenotype=ph, panel=p, symbol=s, gene=gn, acute_phase=s in ACUTE_PHASE,
                                      in_panel=in_panel[j], lit=is_lit[j], nm_up=nm[j], n_pat=int(pn[j]),
                                      n_hc=int(hn[j]), n_case=len(P), detect=det.loc[gn, ph],
                                      hit_samples=';'.join(c['case'][Pg[:, j]])))
    return pd.DataFrame(rows), pd.DataFrame(gene_rows)


t1, g1 = summarize('det10')
t2, g2 = summarize('read10')
tbl_all, genes_all = pd.concat([t1, t2]), pd.concat([g1, g2])
tbl_all.to_csv(OUT / 'disease_tissue_summary.csv', index=False)
genes_all.to_csv(OUT / 'disease_tissue_genes.csv', index=False)
tbl, genes_tbl = t2, g2

key = ['phenotype', 'panel']
cmp = t1[key + ['prior', 'lit_level', 'lit_n', 'lit_detected', 'lit_by_nm', 'nm_up', 'nm_new']].merge(
    t2[key + ['n_panel', 'nm_up', 'nm_new', 'nm_new_single', 'nm_new_non_ap', 'pat_carrier', 'hc_carrier', 'fisher_p']],
    on=key, suffixes=('_det10', '_read10'))

NOVEL_MIN_GENES = 5
NOVEL_MIN_PATIENTS = 3
NOVEL_MAX_TOP_SHARE = 0.5
NOVEL_MIN_RATIO = 2
NOVEL_EXCLUDE = {'neuron', 'oligodendrocyte', 'plasma cell', 'erythroid', 'neutrophil', 'platelet / megakaryocyte', 'T cell'}


def carrier_stats(d):
    s = d['hit_samples'].dropna().str.split(';').explode().value_counts()
    return pd.Series(dict(n_carrier=len(s), top_share=s.iloc[0] / len(d) if len(s) else np.nan))


new_g = genes_tbl[genes_tbl['nm_up'] & genes_tbl['in_panel'] & ~genes_tbl['lit']]
cs = new_g.groupby(key)[['hit_samples']].apply(carrier_stats).reset_index()
cmp = cmp.merge(cs, on=key, how='left').fillna({'n_carrier': 0})
cmp['panel_ratio'] = cmp['pat_carrier'] / cmp['panel'].map(cmp.groupby('panel')['pat_carrier'].median()).clip(lower=0.01)
cmp['novel'] = ((cmp['prior'] == 'off') & ~cmp['panel'].isin(NOVEL_EXCLUDE) & (cmp['lit_level'] == 'none') & (cmp['nm_new_read10'] >= NOVEL_MIN_GENES)
                & (cmp['n_carrier'] >= NOVEL_MIN_PATIENTS) & (cmp['top_share'] <= NOVEL_MAX_TOP_SHARE)
                & (cmp['panel_ratio'] >= NOVEL_MIN_RATIO) & (cmp['fisher_p'] < 0.05))
cmp.to_csv(OUT / 'disease_tissue_compare.csv', index=False)
cmp[cmp['novel'] | cmp['prior'].ne('off')].round(3)

### 4a. 사전 후보 밖에서 데이터가 강하게 가리키는 조직 (novel 후보)

관심 조직 목록(`PRIOR`)에도, 문헌에도 없는데 NM이 여러 마커를 잡은 질병-조직. 결과를 보고 고르는 탐색적 선택이므로 아래 기준을 모두 만족해야 하고, 그림에서도 `Novel` 별로 따로 표시한다.

| 기준 | 값 | 이유 |
|---|---|---|
| NM 신규 유전자 수 | >= `NOVEL_MIN_GENES` (5) | 1-2개짜리는 이야기 거리가 아님 |
| 신규 유전자를 가진 환자 수 | >= `NOVEL_MIN_PATIENTS` (3) | 한두 명의 이상 샘플 배제 |
| 가장 많이 가진 1명의 몫 | <= `NOVEL_MAX_TOP_SHARE` (50%) | 1명이 대부분을 채우는 경우 배제 |
| 환자 보유율 / 같은 패널의 전 질병 중앙값 | >= `NOVEL_MIN_RATIO` (2배) | 모든 질병에서 다 오르는 패널(예: 큰 패널) 배제 |
| 환자 vs 같은 배치 HC 보유율 | 단측 Fisher p < 0.05 | HC에서도 흔한 신호 배제 |
| 패널 | `NOVEL_EXCLUDE`(뇌, 혈구 계열) 제외 | 뇌: 혈뇌장벽 때문에 CNS 손상 없이 혈장에 뇌 유래 RNA가 나올 근거가 약하고 HPA neuron 패널은 망막·고환·근육과 묶인 group enriched 유전자가 많음. 혈구: 조직 신호가 아님 |

이 행들은 C1/C2 근거 없이 데이터로만 고른 것이라 '검증할 가설'이지 결론이 아니다.

In [ ]:
cmp[cmp['novel']][key + ['n_panel', 'nm_new_read10', 'nm_new_single', 'n_carrier', 'top_share', 'pat_carrier',
                          'hc_carrier', 'panel_ratio', 'fisher_p']].round(3)

## 5. 한 그림으로 보기 -- 관심 조직별 문헌 / NM / NM 신규 / 1명 hit

기준은 `read10`(1회성 검출 마커 포함: 1개 샘플이라도 검출, 환자 hit read >= `MIN_HIT_READS`).

이 그림이 한 번에 보여야 하는 세 가지:

| 열 | 질문 |
|---|---|
| **1. Source study** | 이 샘플을 만든 원 연구는 이 조직 신호를 보고했나 (질병 라벨 아래 연구명) |
| **2. Other cfRNA / cfDNA** | 이 데이터와 별개로 다른 cfRNA 연구, cfDNA 조직 기원 연구는 이 조직을 제시했나 |
| **3. This study** (오른쪽 막대, `Literature markers (NM patients)` 열) | 문헌이 이름을 댄 마커를 NM이 몇 명에서 잡았나, 그리고 문헌에 없던 마커를 몇 개 잡았나 |

점 모양: 채운 원 = 유전자 이름까지 보고, 빈 원 = 조직·세포형 수준, 빗금 원 = 일부 유전자만/저자가 조직으로 해석하지 않음/보충자료 미확인.
왼쪽에는 그 조직을 고른 이유(C1 기전, C2 임상 혈액 마커)를 적는다. 혈액 계통 세포형은 Tissue 열의 `Blood / marrow`.
- 행 세 종류 (문헌 마커가 있는 질병-조직은 사전 후보 여부와 무관하게 항상 표시):
  - **사전 후보**: C1 + C2 >= 1을 통과한 관심 조직(keep / cond / blood) 전부, 단 검출 가능한 HPA 마커가 1개 이상(`n_panel > 0`). 검출 결과로 고르지 않는다. 마커가 없어 빠진 조직은 셀 출력에 표시(예: 췌관).
  - **Novel(빨간 별)**: 사전 후보에도 문헌에도 없지만 4a 기준을 모두 통과한 조직. 각 질병 블록의 맨 아래에 놓고 선정 이유 칸은 `not in prior: data-driven`.
- `Patients with new` = NM 신규 유전자를 1개 이상 가진 환자 수. 막대가 커도 이 값이 1-2면 소수 환자 신호다.
- 조건부 조직의 필요 조건과 문헌 방향 메모는 1절 `CONDITION`, `LIT_NOTE`에 있다(그림에서는 생략).
- 오른쪽 막대: 유전자 수, x축은 symlog(0-1 선형, 그 위 로그)라 큰 값이 작은 값을 가리지 않는다. 막대 끝 숫자 = 실제 개수.
- 질병마다 음영을 번갈아 넣었다.

In [ ]:
pri = cmp['prior'].isin(['keep', 'cond', 'blood'])
untestable = cmp[pri & (cmp['n_panel'] == 0)][['phenotype', 'panel']]
order = {p: i for i, p in enumerate(PRIOR)}
v = cmp[(pri & (cmp['n_panel'] > 0)) | cmp['novel'] | (cmp['lit_n'] > 0)].assign(
    o=lambda d: d['phenotype'].map(order), r=lambda d: d['novel'] + 2 * (d['panel'] == 'no tissue given'))
v = v.sort_values(['o', 'r'], kind='stable').reset_index(drop=True)
print('not shown, no detectable HPA marker (n_panel = 0):', list(map(tuple, untestable.values)))
missing = [k for k, nv in zip(zip(v['phenotype'], v['panel']), v['novel'])
           if k not in RATIONALE and not nv and k[1] != 'no tissue given' and not any(m[:2] == k for m in LIT_MOVED)]
assert not missing, missing
keys = list(zip(v['phenotype'], v['panel']))
gl = genes_tbl.set_index(['phenotype', 'panel', 'symbol'])['n_pat']
lit_txt = []
for k in keys:
    def lab(x):
        t = hpa_types(x)
        if k + (x,) in LIT_MOVED:
            tag = '*'
        elif k[1] == 'no tissue given':
            tag = f"[{'/'.join(t)}]" if t else ''
        else:
            tag = '' if k[1] in {panel_of_type.get(c) for c in t} else '°'
        return x + ('↓' if x in LIT_DOWN else '') + tag
    items = [f'{lab(x)}({int(gl[k + (x,)])})' if k + (x,) in gl.index and gl[k + (x,)] > 0
             else (lab(x) if k + (x,) in gl.index else f'{lab(x)}†') for x in LIT_GENES.get(k, ([], ''))[0]]
    lit_txt.append(textwrap.fill(', '.join(items), 52))
row_h = np.array([max(1.0, 0.42 * (t.count(chr(10)) + 1)) for t in lit_txt])
y = -(np.cumsum(row_h) - row_h / 2)
y = y - y.min() + row_h[-1] / 2
groups = [(ph, y[(v['phenotype'] == ph).values]) for ph in dict.fromkeys(v['phenotype'])]
half = pd.Series(row_h / 2, index=y)
bars = [('lit_n', OI['grey'], 'Literature-named markers'),
        ('nm_up_read10', OI['orange'], 'NM (patients, no HC)'),
        ('nm_new_read10', OI['vermillion'], 'NM new (not in literature)'),
        ('nm_new_single', OI['purple'], 'NM new, single patient')]
ev_cols = [('source', '1. Source\nstudy'), ('cfrna', '2. Other\ncfRNA'), ('cfdna', '2. Other\ncfDNA')]

fig = plt.figure(figsize=(29, 0.45 * row_h.sum() + 2))
gs = fig.add_gridspec(1, 2, width_ratios=[5.2, 2], wspace=0.02)
at = fig.add_subplot(gs[0])
ab = fig.add_subplot(gs[1], sharey=at)
for k, (ph, yy) in enumerate(groups):
    if k % 2 == 0:
        for ax in (at, ab):
            ax.axhspan(yy.min() - half[yy.min()], yy.max() + half[yy.max()], color='0.93', zorder=0, lw=0)
    at.text(0, yy.mean(), f"{ph}\n({len(cohort[ph]['case'])}/{len(cohort[ph]['hc'])})\n{SOURCE_STUDY[ph].replace('; ', chr(10))}", va='center',
            fontsize=FS_ANNOT_SM + 1, fontweight='bold', linespacing=1.3)
X_ORG, X_TIS, X_MEC, X_CLI, X_DOT, X_NOV, X_PAT, X_LIT = 6.6, 10.2, 14.4, 21.4, [27.8, 30.3, 32.8], 35.0, 37.4, 39.4
for yi, k, nv, npt, lt in zip(y, keys, v['novel'], v['n_carrier'], lit_txt):
    at.text(X_ORG, yi, TISSUE_OF[k[1]], va='center', fontsize=FS_ANNOT_SM + 1)
    at.text(X_TIS, yi, 'unassigned' if k[1] == 'no tissue given' else k[1], va='center', fontsize=FS_ANNOT_SM + 1)
    mec, cli = RATIONALE.get(k, ('source markers, unassigned', '-') if k[1] == 'no tissue given'
                             else ('source markers, HPA-placed', '-') if any(m[:2] == k for m in LIT_MOVED)
                             else ('not in prior: data-driven', '-'))
    at.text(X_MEC, yi, mec, va='center', fontsize=FS_ANNOT_SM, style='italic' if nv else 'normal')
    at.text(X_CLI, yi, cli, va='center', fontsize=FS_ANNOT_SM)
    if nv:
        at.scatter(X_NOV, yi, s=110, marker='*', color=OI['vermillion'], ec='k', lw=0.4, zorder=2)
    at.text(X_PAT, yi, f'{int(npt)}', ha='center', va='center', fontsize=FS_ANNOT_SM + 1)
    at.text(X_LIT, yi, lt, va='center', fontsize=FS_ANNOT_SM - 1)
    for xd, (col, _) in zip(X_DOT, ev_cols):
        lvl = EVIDENCE.get(k, {}).get(col)
        if lvl == 'genes':
            at.scatter(xd, yi, s=45, color='k', zorder=2)
        elif lvl == 'tissue':
            at.scatter(xd, yi, s=45, facecolor='w', edgecolor='k', lw=1.2, zorder=2)
        elif lvl == 'partial':
            at.scatter(xd, yi, s=45, facecolor='w', edgecolor='k', lw=1.2, hatch='////', zorder=2)
for xd, (_, name) in zip(X_DOT, ev_cols):
    at.text(xd, y.max() + 1, name, ha='center', va='bottom', fontsize=FS_ANNOT_SM + 1, fontweight='bold')
for x, h in [(0, 'Disease (n pt/HC)\nsource study'), (X_ORG, 'Tissue'), (X_TIS, 'Cell type'),
             (X_MEC, 'Why chosen: C1 mechanism'), (X_CLI, 'C2 clinical blood marker')]:
    at.text(x, y.max() + 1, h, va='bottom', fontsize=FS_ANNOT_SM + 1, fontweight='bold')
at.text(X_LIT, y.max() + 1, 'Literature markers\n(NM patients; ↓ reported down; † not modelled;\n* placed by HPA; ° not HPA-enriched here; [HPA type])', va='bottom', fontsize=FS_ANNOT_SM + 1, fontweight='bold')
for x, h in [(X_NOV, 'Novel\ntissue'), (X_PAT, 'Patients\nwith new')]:
    at.text(x, y.max() + 1, h, ha='center', va='bottom', fontsize=FS_ANNOT_SM + 1, fontweight='bold')
at.text(X_DOT[1], y.max() + 2.3, 'Reported in literature', ha='center', va='bottom', fontsize=FS_ANNOT_SM + 1, fontweight='bold')
for mk, lab, xo in [(dict(color='k'), 'gene names', 0), (dict(facecolor='w', edgecolor='k', lw=1.2), 'tissue level', 4.5),
                    (dict(facecolor='w', edgecolor='k', lw=1.2, hatch='////'), 'partial / cell type by HPA', 9.5)]:
    at.scatter(X_MEC + xo, y.min() - 1.4, s=45, **mk)
    at.text(X_MEC + xo + 0.5, y.min() - 1.4, lab, va='center', fontsize=FS_ANNOT_SM)
at.set_xlim(-0.1, 50.5)
at.set_ylim(y.min() - 2, y.max() + 3.2)
at.axis('off')

w = 0.8 / len(bars)
for k, (col, c, name) in enumerate(bars):
    yy = y + (len(bars) / 2 - 0.5 - k) * w
    ab.barh(yy, v[col], height=w, color=c, label=name, zorder=2)
    for yi, n in zip(yy, v[col]):
        if n > 0:
            ab.text(n * 1.08 + 0.05, yi, str(int(n)), va='center', fontsize=FS_ANNOT_SM - 1, color=c)
ab.set_xscale('symlog', linthresh=1)
ab.set_xlim(0, v[[b[0] for b in bars]].values.max() * 2.5)
ab.set_xticks([0, 1, 2, 5, 10, 20, 50])
ab.set_xticklabels(['0', '1', '2', '5', '10', '20', '50'])
ab.set_xlabel(f'Marker genes (symlog; any detection, NM hit reads >= {MIN_HIT_READS})')
ab.tick_params(axis='y', left=False, labelleft=False)
ab.spines['left'].set_visible(False)
ab.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False,
          title='3. This study: normative model', title_fontproperties=dict(weight='bold'))
save_fig(fig, 'lit_nm_by_tissue')
plt.show()

## 6. 세포형 마커 enrichment (질병 x 패널, 표준 ORA)

질병마다 환자 전체의 신호를 한 유전자 집합으로 모아 HPA 패널과 over-representation analysis(ORA)를 한다.

- 신호 집합 = 그 질병 환자 1명 이상에서 상승(BH q < 0.05, read >= `MIN_HIT_READS`)하고 같은 배치 HC에서는 0명인 유전자 (engine 유전자 전체 대상).
- universe = 그 질병의 비교 샘플(환자 + 같은 배치 HC)에서 1번이라도 검출된 engine 유전자.
- 패널 = 2절 HPA 패널(혈장 특이성 필터 후) ∩ universe.
- 단측 Fisher exact(over-representation), 전체 칸 BH 보정. 기대 = 신호 수 x 패널 크기 / universe 크기.
- 그림: 행 = 세포형 패널(조직별로 묶고 음영 교대), 열 = 질병. 점 크기 = 신호 집합 ∩ 패널 유전자 수, 색 = -log10(BH q).
  점선 원 = 사전 관심 조직(`PRIOR`의 keep/cond/blood, 관측 0이면 빈 점선 원만).
- 한계: 검출률 매칭이 없어 혈장에 원래 많은 조직(간, 혈구)이 유리할 수 있고, 유전자 간 상관을 무시한다. novel 판정(4a)에는 쓰지 않는다.

In [ ]:
ALL_CNT_CACHE = OUT / 'all_counts.npz'
if ALL_CNT_CACHE.exists():
    C_all = sp.load_npz(ALL_CNT_CACHE).tocsr()
else:
    A = ad.read_h5ad(config.H5AD_PATH, backed='r')
    si = A.obs_names.get_indexer(names)
    o = np.argsort(si)
    X = sp.csr_matrix(A.X[si[o]])[:, A.var_names.get_indexer(genes)]
    C_all = X[np.argsort(o)].tocsr()
    sp.save_npz(ALL_CNT_CACHE, C_all)

pn_order = [p for p in PANELS if PANELS[p]]
ens_set = {p: set(panel_tbl.loc[panel_tbl['panel'] == p, 'col']) for p in pn_order}
enr = []
for ph, c in cohort.items():
    rc, rh = row_of[c['case']].values, row_of[c['hc']].values
    universe = np.asarray((C_all[np.concatenate([rc, rh])] > 0).sum(0)).ravel() > 0
    hit = (UP[rc] & (C_all[rc] >= MIN_HIT_READS).toarray()).any(0) & ~UP[rh].any(0) & universe
    n_u, n_s = universe.sum(), hit.sum()
    for p in pn_order:
        pm = np.zeros(len(genes), bool)
        pm[list(ens_set[p])] = True
        pm &= universe
        k, n_p = int((hit & pm).sum()), int(pm.sum())
        enr.append(dict(phenotype=ph, panel=p, prior=PRIOR[ph].get(p, 'off'), obs=k, exp=n_s * n_p / n_u,
                        n_signal=n_s, n_panel=n_p, n_universe=n_u,
                        p=fisher_exact([[k, n_p - k], [n_s - k, n_u - n_p - n_s + k]], alternative='greater')[1]))
enr = pd.DataFrame(enr)
enr['q'] = false_discovery_control(enr['p'])
enr['log2_oe'] = np.log2((enr['obs'] + 0.5) / (enr['exp'] + 0.5))
enr.to_csv(OUT / 'disease_tissue_enrichment.csv', index=False)
print(enr.groupby('phenotype')[['n_signal', 'n_universe']].first().to_string())

ph_order = list(PRIOR)
tis_order = list(dict.fromkeys(TISSUE_OF[p] for p in pn_order))
pn_rows = sorted(pn_order, key=lambda p: tis_order.index(TISSUE_OF[p]))
d = enr.assign(x=enr['phenotype'].map(ph_order.index), y=enr['panel'].map(pn_rows.index))
size = lambda n: 72 * np.sqrt(np.maximum(n, 1))
fig, ax = plt.subplots(figsize=(0.62 * len(ph_order) + 7, 0.42 * len(pn_rows) + 3))
X0 = -0.36
for k, t in enumerate(tis_order):
    yy = [pn_rows.index(p) for p in pn_rows if TISSUE_OF[p] == t]
    if k % 2 == 0:
        ax.add_patch(Rectangle((X0, min(yy) - 0.5), 1 - X0, len(yy), transform=ax.get_yaxis_transform(),
                               color='0.92', lw=0, zorder=0, clip_on=False))
    ax.text(X0 + 0.01, np.mean(yy), t, transform=ax.get_yaxis_transform(), ha='left', va='center',
            fontsize=FS_ANNOT_SM + 5, fontweight='bold')
dd = d[d['obs'] > 0]
sc = ax.scatter(dd['x'], dd['y'], s=size(dd['obs']), c=-np.log10(dd['q']), cmap='Reds', vmin=0,
                vmax=max(2, -np.log10(dd['q']).max()), edgecolors='0.5', linewidths=0.3, zorder=2)
pr = d[d['prior'] != 'off']
ax.scatter(pr['x'], pr['y'], s=size(pr['obs']) * 1.6, facecolors='none', edgecolors='k', linewidths=1.1,
           linestyles='--', zorder=3)
ax.set_xticks(range(len(ph_order)))
ax.set_xticklabels([f"{p} ({len(cohort[p]['case'])}/{len(cohort[p]['hc'])})" for p in ph_order], rotation=90)
ax.set_yticks(range(len(pn_rows)))
ax.set_yticklabels(pn_rows)
ax.tick_params(axis='y', length=0)
ax.set_xlim(-0.5, len(ph_order) - 0.5)
ax.set_ylim(len(pn_rows) - 0.5, -0.5)
cb = plt.colorbar(sc, ax=ax, fraction=0.03, shrink=0.35, anchor=(0, 1), label='-log10 BH q')
cb.ax.axhline(-np.log10(0.05), color='k', lw=1)
for n in [1, 10, 50]:
    ax.scatter([], [], s=size(n), c='0.6', label=f'{n} NM genes')
ax.scatter([], [], s=size(10) * 1.6, facecolors='none', edgecolors='k', linewidths=1.1, linestyles='--',
           label='prior tissue of interest')
ax.legend(loc='lower left', bbox_to_anchor=(1.01, 0), frameon=False, labelspacing=1.6)
plt.tight_layout()
save_fig(fig, 'disease_tissue_enrichment')
plt.show()
enr[enr['p'] < 0.05].sort_values('p').round(3)

## 7. 문헌 마커는 각각 어디서 잡혔나

문헌 마커는 모두 검출률 10% 이상이라 두 기준에서 결과가 같다 (`read10`만 표시).

In [ ]:
genes_tbl[genes_tbl['lit']][['phenotype', 'panel', 'symbol', 'in_panel', 'nm_up',
                             'n_pat', 'n_case', 'n_hc', 'detect']].round(3)

## 8. 1명 hit 유전자 -- 그룹 분석으로는 보이지 않는 샘플 수준 사례

대상: `read10` 기준, 관심 조직(keep/cond) 패널 유전자 중 **환자 1명에서만** BH 상승(read >= `MIN_HIT_READS`), 같은 배치 HC 0명, 문헌이 이름을 댄 마커 아님.

| 열 | 의미 |
|---|---|
| `reads`, `cpm`, `Z` | 그 환자의 원시 read, CPM, NM Z |
| `hc_pool_detect` | 전체 HC pool(592명)에서 read > 0 비율 |
| `hc_pool_max_cpm` | 전체 HC pool 최대 CPM. 환자 CPM이 이보다 크면 건강인 범위 밖 |
| `pat_n_up` / `pat_n_up_hc_pct` | 그 환자의 전체 상승 유전자 수, HC pool 분포에서의 백분위. 높으면 '전반적으로 튀는 샘플'의 일부일 가능성 |

In [ ]:
prior_cells = set(map(tuple, tbl.loc[tbl['prior'].isin(['keep', 'cond']), key].values))
hc_up = n_up.loc[hm['sample']].values
s1 = genes_tbl[genes_tbl['nm_up'] & genes_tbl['in_panel'] & ~genes_tbl['lit']
               & (genes_tbl['n_pat'] == 1)]
s1 = s1[[k in prior_cells for k in zip(s1['phenotype'], s1['panel'])]].copy()
s1['sample'] = s1['hit_samples']
s1['reads'] = [cnt.at[s, g] for s, g in zip(s1['sample'], s1['gene'])]
s1['cpm'] = [cpm.at[s, g] for s, g in zip(s1['sample'], s1['gene'])]
s1['Z'] = [Z_all[row_of[s], pd.Index(genes).get_loc(g)] for s, g in zip(s1['sample'], s1['gene'])]
s1['hc_pool_detect'] = [(cnt.loc[hm['sample'], g] > 0).mean() for g in s1['gene']]
s1['hc_pool_max_cpm'] = [cpm.loc[hm['sample'], g].max() for g in s1['gene']]
s1['pat_n_up'] = n_up.loc[s1['sample']].values
s1['pat_n_up_hc_pct'] = [(hc_up < x).mean() * 100 for x in s1['pat_n_up']]
s1 = s1.sort_values(['phenotype', 'panel', 'reads'], ascending=[True, True, False])
s1.to_csv(OUT / 'single_patient_hits.csv', index=False)
s1[['phenotype', 'panel', 'symbol', 'acute_phase', 'sample', 'reads', 'cpm', 'Z', 'detect', 'hc_pool_detect',
    'hc_pool_max_cpm', 'pat_n_up', 'pat_n_up_hc_pct']].round(2)

### 8a. 1명 hit 유전자: 관측 HC pool vs 모형이 예측한 HC pool vs 그 환자의 조건부 분포

**선정 기준** (8절 표와 동일, 한 유전자 = 한 칸)

| 조건 | 값 |
|---|---|
| 조직 | 그 질병의 관심 조직 패널(keep/cond), HPA 공식 cell type / group enriched |
| hit | 환자 1명에서만 샘플 내 BH q < 0.05, Z > 0, read >= `MIN_HIT_READS` |
| 대조 | 같은 배치 HC에서 BH 상승 0명 |
| 신규 | 문헌이 이름을 댄 마커 아님 |
| 표시 | 질병마다 그림 1장: `hit_ratio`(환자 read / 조건부 분포 99.5% 분위수) 상위 `N_PER`개, 패널별로 묶음 |

**그림 구성** (질병마다 1장, x = hit 유전자(괄호 = 환자 P1, P2, ...), y = log10(CPM + 1))
- 색 = 환자 (violin과 별이 같은 색, 오른쪽 Patient 범례에 SRR 번호).
- violin = 그 환자의 공변량에서 engine이 예측한 건강 분포(NB + 배치 random effect `tau2`, `N_DRAW`회 추출).
- 검은 가로선 = 그 조건부 분포의 99.5% 분위수.
- 회색 점 = 같은 질병의 모든 환자의 관측값 (다른 환자들은 그 유전자가 대부분 0인지 보여준다).
- 별 = hit 환자의 관측값. 별이 violin과 회색 점 위로 떨어져 있을수록 그 한 명만 벗어난 것이다.
- 모형 보정 확인(HC 관측 vs 예측 검출률)은 아래 PPC 그림.

In [ ]:
from numpy.polynomial.hermite_e import hermegauss
from scipy.stats import nbinom

from MixedEffectsModeling.core.model_engine_mixed import NormativeModelEngineMixed
from MixedEffectsModeling.validation.lobo_engine import load_full_data


def nb_params(engine, X, g):
    rec = engine.genes[g]
    Xa = np.column_stack([np.ones(len(X)), X])
    if rec.route == 'pool':
        rg = engine.rare_glm
        mult = np.exp(X @ rg['beta'][1:])
        if rg.get('mult_lo') is not None:
            mult = np.clip(mult, rg['mult_lo'], rg['mult_hi'])
        mu = (rec.mean_hc + rg['eps']) * np.exp(rg['beta'][0]) * mult
        alpha = np.full(len(X), 1e-8 if rg['family'] == 'poisson' else rg['alpha'])
        tau2 = 0.0 if rg['family'] == 'poisson' else rg.get('tau2', 0.0)
    else:
        mu = np.exp(Xa @ np.nan_to_num(rec.mu_coef, nan=0.0))
        if not np.all(np.isnan(rec.disp_coef)):
            alpha = np.exp(-Xa @ np.nan_to_num(rec.disp_coef, nan=0.0))
        elif rec.trend_alpha is not None:
            alpha = np.full(len(X), rec.trend_alpha)
        else:
            alpha = np.full(len(X), engine.alpha_fn(float(mu.mean())))
        tau2 = rec.tau2
    return np.clip(mu, 1e-6, 1e8), alpha, float(tau2), rec.route


COND_CACHE = OUT / 'single_hit_conditional.pkl'
if COND_CACHE.exists():
    cond, hc_par = pickle.load(open(COND_CACHE, 'rb'))
else:
    engine = NormativeModelEngineMixed.load(config.ENGINE_MIXED_DIR)
    data = load_full_data()
    pos = pd.Series(np.arange(len(data['names'])), index=data['names'])
    ip, ih = pos[s1['sample']].values, pos[hm['sample']].values
    Xp = engine.scaler.transform(engine.covariates.transform(data['obs'][ip], data['Y'][ip]))
    Xh = engine.scaler.transform(engine.covariates.transform(data['obs'][ih], data['Y'][ih]))
    rows, hc_par = [], {}
    for k, (smp, g) in enumerate(zip(s1['sample'], s1['gene'])):
        mu, alpha, tau2, route = nb_params(engine, Xp[k:k + 1], g)
        rows.append(dict(sample=smp, gene=g, route=route, mu=mu[0], alpha=alpha[0], tau2=tau2))
        if g not in hc_par:
            hc_par[g] = nb_params(engine, Xh, g)[:3]
    cond = pd.DataFrame(rows)
    pickle.dump((cond, hc_par), open(COND_CACHE, 'wb'))


def cond_cdf(y, mu, alpha, tau2, n_nodes=7):
    nodes, w = hermegauss(n_nodes)
    w = w / w.sum()
    n = 1.0 / alpha
    sd = np.sqrt(max(tau2, 0.0))
    return sum(wi * nbinom.cdf(y, n, n / (n + mu * np.exp(sd * nd))) for nd, wi in zip(nodes, w))


def cond_draw(mu, alpha, tau2, rng):
    n = 1.0 / np.asarray(alpha, dtype=float)
    m = np.asarray(mu, dtype=float) * np.exp(rng.normal(0, np.sqrt(max(tau2, 0.0)), np.shape(mu)))
    return rng.negative_binomial(n, n / (n + m))


s1 = s1.drop(columns=[c for c in ['mu', 'alpha', 'tau2', 'route'] if c in s1]).merge(cond, on=['sample', 'gene'], how='left')
s1['cond_hi'] = [int(np.argmax(cond_cdf(np.arange(int(max(r.reads, 50)) + 1), r.mu, r.alpha, r.tau2) >= 0.995))
                 for r in s1.itertuples()]
s1['cond_tail_p'] = [1 - cond_cdf(r.reads - 1, r.mu, r.alpha, r.tau2) for r in s1.itertuples()]
s1['hit_ratio'] = s1['reads'] / np.maximum(s1['cond_hi'], 1)
s1['above_hc_pool_max'] = s1['cpm'] > s1['hc_pool_max_cpm']
s1['hc_detect_obs'] = s1['hc_pool_detect']
s1['hc_detect_pred'] = [float(np.mean(1 - cond_cdf(0, *hc_par[g]))) for g in s1['gene']]
s1['hc_detect_ratio'] = s1['hc_detect_pred'] / s1['hc_detect_obs'].clip(lower=1e-3)
s1.to_csv(OUT / 'single_patient_hits.csv', index=False)
print('HC detection, predicted / observed (median over hit genes):', round(s1['hc_detect_ratio'].median(), 2))
s1[['phenotype', 'panel', 'symbol', 'sample', 'reads', 'mu', 'cond_hi', 'hit_ratio', 'cond_tail_p', 'Z',
    'hc_detect_obs', 'hc_detect_pred', 'above_hc_pool_max']].sort_values('hit_ratio', ascending=False).round(3).head(30)

In [ ]:
N_PER = 30
N_DRAW = 2000
rng = np.random.default_rng(0)
for ph in PRIOR:
    sub = s1[s1['phenotype'] == ph].sort_values('hit_ratio', ascending=False).head(N_PER)
    if sub.empty:
        continue
    sub = sub.sort_values(['panel', 'hit_ratio'], ascending=[True, False]).reset_index(drop=True)
    pid = {x: f'P{i + 1}' for i, x in enumerate(dict.fromkeys(sub.sort_values('sample')['sample']))}
    cases = cohort[ph]['case']
    pal = [OI[k] for k in ['orange', 'blue', 'vermillion', 'green', 'purple', 'sky', 'yellow', 'black']]
    if len(pid) > len(pal):
        pal = [plt.cm.tab20(i % 20) for i in range(len(pid))]
    col = {x: pal[i] for i, x in enumerate(pid)}
    n_slot = max(len(sub), 8)
    pad = (n_slot - len(sub)) / 2
    fig, ax = plt.subplots(figsize=(0.5 * n_slot + 2, 5))
    viol = [np.log10(cond_draw(np.full(N_DRAW, r.mu), np.full(N_DRAW, r.alpha), r.tau2, rng) / libsize[r.sample] * 1e6 + 1)
            + rng.normal(0, 1e-3, N_DRAW) for r in sub.itertuples()]
    vp = ax.violinplot(viol, positions=range(len(sub)), widths=0.8, showextrema=False, bw_method=0.3)
    for b, r in zip(vp['bodies'], sub.itertuples()):
        b.set_facecolor(col[r.sample])
        b.set_edgecolor('none')
        b.set_alpha(0.35)
    for j, r in enumerate(sub.itertuples()):
        y = np.log10(cpm.loc[cases, r.gene].values + 1)
        ax.scatter(j + rng.uniform(-0.18, 0.18, len(y)), y, s=5, color=OI['grey'], alpha=0.7, lw=0, zorder=2)
        hi = np.log10(r.cond_hi / libsize[r.sample] * 1e6 + 1)
        ax.plot([j - 0.3, j + 0.3], [hi, hi], color='k', lw=1, zorder=3)
        ax.scatter([j], [np.log10(r.cpm + 1)], s=110, marker='*', color=col[r.sample], ec='k', lw=0.5, zorder=4)
    for k, (pn, g) in enumerate(sub.groupby('panel', sort=False)):
        if k % 2 == 0:
            ax.axvspan(g.index.min() - 0.5, g.index.max() + 0.5, color='0.93', lw=0, zorder=0)
        ax.text(g.index.to_series().mean(), 1.01, textwrap.fill(pn, max(6, 4 * len(g)), break_long_words=False), transform=ax.get_xaxis_transform(), ha='center', va='bottom',
                fontsize=FS_ANNOT_SM + 1, fontweight='bold')
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([f'{r.symbol} ({pid[r.sample]})' for r in sub.itertuples()], rotation=90)
    ax.set_xlim(-0.6 - pad, len(sub) - 0.4 + pad)
    ax.set_ylabel('log10(CPM + 1)')
    hs = [plt.Line2D([], [], marker='*', ls='', ms=11, mfc='w', mec='k', mew=0.5),
          plt.Rectangle((0, 0), 1, 1, color='0.6', alpha=0.35), plt.Line2D([], [], color='k', lw=1),
          plt.Line2D([], [], marker='o', ls='', ms=3.5, color=OI['grey'])]
    lb = ['hit patient, observed', 'healthy expectation\n(that patient)', 'conditional 99.5%', f'all patients (n = {len(cases)})']
    hs += [plt.Line2D([], [], marker='*', ls='', ms=11, mfc=col[x], mec='k', mew=0.5) for x in pid]
    lb += [f'{pid[x]} {x}' for x in pid]
    ax.legend(hs, lb, loc='upper left', bbox_to_anchor=(1.01, 1.1), frameon=False, fontsize=FS_ANNOT_SM,
              ncol=1 + len(lb) // 12)
    n_all = (s1['phenotype'] == ph).sum()
    ax.set_title(f'{ph}: single-patient tissue hits ({len(sub)} of {n_all}, top by reads / conditional 99.5%)',
                 fontsize=FS_ANNOT_SM + 2, fontweight='bold', pad=34)
    plt.tight_layout()
    save_fig(fig, 'single_patient_hits_strip_' + ''.join(ch if ch.isalnum() or ch == '-' else '_' for ch in ph.replace('+', 'plus')))
    plt.show()

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(s1['hc_detect_obs'], s1['hc_detect_pred'], s=18, color=np.where(s1['above_hc_pool_max'], OI['vermillion'], OI['grey']),
           ec='k', lw=0.3)
lim = max(s1['hc_detect_obs'].max(), s1['hc_detect_pred'].max()) * 1.05
ax.plot([0, lim], [0, lim], ls='--', color='k', lw=0.8)
ax.set_xlabel('HC pool detection rate, observed')
ax.set_ylabel('HC pool detection rate, model-predicted')
ax.set_title('Single-patient hit genes (red: patient above HC pool max)', fontsize=FS_ANNOT_SM)
plt.tight_layout()
save_fig(fig, 'single_patient_hits_hc_ppc')
plt.show()